# 05 — Weak-to-Strong Generalisation

Can strong models be aligned using weak supervisors? The superalignment hypothesis.

## Superalignment Hypothesis

The **superalignment** challenge (OpenAI, 2023): how do we supervise a model that is smarter than us?

**Weak-to-strong generalisation** (Burns et al., 2023): if a strong model is fine-tuned on labels from a weak supervisor, does it:
a) Learn the weak supervisor's errors (floor)
b) Generalise to the weak supervisor's intended knowledge (performance gap recovery)

**Experimental finding**: GPT-4 fine-tuned on GPT-2-generated labels recovered ~80% of the performance gap between GPT-2 and GPT-4 direct training. This suggests strong models can 'see through' weak supervisor errors when they have the underlying capability.

**Bootstrapping**: this enables:
1. Use humans to supervise small models
2. Use those models to help supervise larger models
3. Iterate toward superintelligence

**Caveats**:
- Works better for tasks where the strong model has clear internal representations
- Does not work for tasks requiring genuine knowledge the strong model lacks
- Deceptive alignment is still possible: model could 'generalise' in undesired ways

**Practical implication**: weak supervisors (including humans) may be sufficient for alignment even for superhuman models, if those models have learned good representations during pretraining.

In [ ]:
# W2SG experiment: small supervisor -> large student
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_classification

torch.manual_seed(42)
np.random.seed(42)

# Task: binary classification with ground truth labels
X, y = make_classification(n_samples=3000, n_features=20, n_informative=12,
                            n_redundant=4, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

# Split
X_train, y_train = X[:1600], y[:1600]
X_val, y_val = X[1600:2000], y[1600:2000]
X_test, y_test = X[2000:], y[2000:]

def make_model(capacity='weak'):
    if capacity == 'weak':
        return nn.Sequential(nn.Linear(20, 16), nn.ReLU(), nn.Linear(16, 1))
    elif capacity == 'strong':
        return nn.Sequential(
            nn.Linear(20, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

def train(model, X_tr, y_tr, n_epochs=200, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(n_epochs):
        out = model(X_tr).squeeze(-1)
        loss = nn.BCEWithLogitsLoss()(out, y_tr)
        opt.zero_grad(); loss.backward(); opt.step()
    return model

def accuracy(model, X_data, y_data):
    with torch.no_grad():
        preds = (model(X_data).squeeze(-1) > 0).float()
        return (preds == y_data).float().mean().item()

# Step 1: Train weak supervisor on true labels
weak = make_model('weak')
train(weak, X_train, y_train)
weak_acc = accuracy(weak, X_test, y_test)
print(f'Weak supervisor accuracy: {weak_acc:.3f}')

# Step 2: Generate weak labels for strong model
with torch.no_grad():
    weak_labels = (weak(X_train).squeeze(-1) > 0).float()
weak_label_acc = (weak_labels == y_train).float().mean().item()
print(f'Weak label quality: {weak_label_acc:.3f}')

# Step 3: Train strong model on weak labels (W2SG)
strong_w2sg = make_model('strong')
train(strong_w2sg, X_train, weak_labels)  # supervised by weak
w2sg_acc = accuracy(strong_w2sg, X_test, y_test)

# Step 4: Train strong model on true labels (ceiling)
strong_oracle = make_model('strong')
train(strong_oracle, X_train, y_train)  # supervised by oracle
oracle_acc = accuracy(strong_oracle, X_test, y_test)

print(f'\nResults:')
print(f'  Weak supervisor accuracy: {weak_acc:.3f}')
print(f'  Strong (W2SG, weak labels): {w2sg_acc:.3f}')
print(f'  Strong (oracle, true labels): {oracle_acc:.3f}')

gap_recovered = (w2sg_acc - weak_acc) / max(oracle_acc - weak_acc, 1e-6)
print(f'  Gap recovery: {gap_recovered:.1%}')
print(f'  (Higher % = strong model generalises beyond weak supervisor errors)')